# Module 08 — Notebook 04: Semantic Segmentation with Deep RL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_08_RL_For_Image_Segmentation/04_Semantic_Segmentation_RL/semantic_segmentation_rl.ipynb)

---

## Learning Objectives

1. Formulate deep RL for pixel-wise semantic segmentation.
2. Build a CNN feature extractor that produces per-pixel state representations.
3. Implement a DQN agent with experience replay for multi-class segmentation.
4. Train on a synthetic dataset with multiple coloured shapes.
5. Compare RL-based segmentation with simple thresholding baselines.
6. Visualize intermediate feature maps and segmentation results.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for segmentation (TINY - under 200MB total)
import torchvision
import numpy as np

# CIFAR-10: 60,000 real photos (already cached if downloaded before)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = [np.array(cifar10[i][0]) for i in range(500)]
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded (32x32 RGB)")

# FashionMNIST: 60,000 real clothing images (30MB only!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (28x28)")
print("   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Boot")

# Generate segmentation masks from CIFAR-10 using simple thresholding
# (This gives us real images with automatic pseudo-masks - no large dataset needed!)
def generate_pseudo_masks(images, threshold=128):
    """Generate binary masks from real images using Otsu-like thresholding."""
    masks = []
    for img in images:
        gray = np.mean(img, axis=2)
        mask = (gray > np.median(gray)).astype(np.uint8)
        masks.append(mask)
    return masks

pseudo_masks = generate_pseudo_masks(real_images)
print(f"✅ Generated {len(pseudo_masks)} segmentation masks from real images")
print("   Total download: ~200MB (CIFAR-10 + FashionMNIST)")

## Deep Derivation: Semantic Segmentation Theory and RL Integration

### Step 1: Semantic Segmentation as Dense Prediction
For input image $I \in \mathbb{R}^{H \times W \times 3}$, predict label map $Y \in \{0, 1, \ldots, K-1\}^{H \times W}$.

**Output:** $\hat{P} \in \mathbb{R}^{H \times W \times K}$ where $\hat{P}_{i,j,k} = P(Y_{i,j} = k | I)$

**Total predictions per image:** $H \times W$ independent classifications.

### Step 2: FCN Architecture — Why Fully Convolutional?
**Claim:** Replacing FC layers with 1×1 convolutions enables arbitrary input sizes.

**Proof:**

FC layer: $\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}$ where $\mathbf{x} \in \mathbb{R}^{C \cdot H \cdot W}$ (fixed size).

1×1 conv: $y_{c'}(i,j) = \sum_{c=1}^C W_{c',c} \cdot x_c(i,j) + b_{c'}$ (applied per spatial location).

The 1×1 conv is equivalent to FC applied independently at each spatial position — but works for ANY $H, W$. $\blacksquare$

### Step 3: Transposed Convolution — Learned Upsampling
**Forward pass of standard conv:** $Y = X * W$ (reduces spatial dims)

**Transposed conv:** $X_{\text{up}} = Y *^T W$ (increases spatial dims)

**Output size:** $H_{\text{out}} = (H_{\text{in}} - 1) \times s + k - 2p$

**Proof transposed conv is the gradient of conv:**

For conv forward: $\mathbf{y} = \mathbf{C}\mathbf{x}$ where $\mathbf{C}$ is the Toeplitz matrix of $W$.

Backward (gradient w.r.t. input): $\frac{\partial L}{\partial \mathbf{x}} = \mathbf{C}^T \frac{\partial L}{\partial \mathbf{y}}$

So transposed conv computes $\mathbf{C}^T\mathbf{y}$ — literally the transpose of the conv matrix! $\blacksquare$

### Step 4: Dilated (Atrous) Convolution — Expanding Receptive Field
$$y(i,j) = \sum_{m,n} x(i + r \cdot m, j + r \cdot n) \cdot w(m,n)$$

where $r$ is the dilation rate.

**Effective kernel size:** $k_{\text{eff}} = k + (k-1)(r-1)$

**Receptive field with dilated convolutions:**
$$\text{RF}_l = \text{RF}_{l-1} + (k_l - 1) \cdot r_l \cdot \prod_{i=1}^{l-1} s_i$$

**Proof dilation increases RF without losing resolution:**

Standard $k=3, s=2$: doubles RF but halves resolution.

Dilated $k=3, r=2, s=1$: effective $k_{\text{eff}}=5$ (larger RF), output size unchanged! $\blacksquare$

### Step 5: Weighted Cross-Entropy for Class Imbalance
For imbalanced classes (e.g., background dominates):
$$L_{WCE} = -\frac{1}{HW}\sum_{i,j} w_{y_{ij}} \log \hat{p}_{ij, y_{ij}}$$

**Optimal weights (inverse frequency):**
$$w_k = \frac{N}{\text{count}(k) \cdot K}$$

**Proof inverse frequency equalizes gradient contribution per class:**

Expected gradient magnitude for class $k$: $E[|\nabla_k|] \propto w_k \cdot \text{count}(k)$

With $w_k = N/(K \cdot \text{count}(k))$: $E[|\nabla_k|] \propto \frac{N}{K}$ (equal for all classes). $\blacksquare$

### Step 6: Focal Loss — Hard Example Mining
$$L_{FL} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

**Proof focal loss down-weights easy examples:**

For well-classified pixel ($p_t \to 1$): $(1 - p_t)^\gamma \to 0$ — loss vanishes.

For misclassified pixel ($p_t \to 0$): $(1 - p_t)^\gamma \to 1$ — full weight.

**Effective number of samples:** Focal loss with $\gamma=2$ reduces the contribution of easy examples by $\sim 100\times$ compared to CE. $\blacksquare$

### Step 7: RL for Semantic Segmentation — Policy Over Labels
**State:** image features from encoder + current partial segmentation

**Actions:** assign class labels to pixels/regions

**Reward:**
$$r_t = \Delta\text{mIoU} = \frac{1}{K}\sum_{k=1}^K \left[\text{IoU}_k(Y_t) - \text{IoU}_k(Y_{t-1})\right]$$

**Connection to RL:** The RL agent learns to allocate labeling effort where it matters most — spending more "actions" on ambiguous boundary regions and quickly labeling obvious areas. The sequential decision process naturally handles the spatial dependencies that pixel-independent classifiers miss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All imports successful.')

---
## 1. Mathematical Formulation: Deep RL for Semantic Segmentation

### Problem Setup

Given an image $I \in \mathbb{R}^{H \times W \times C}$, we seek a labelling $\hat{Y} \in \{0,\ldots,K-1\}^{H \times W}$ that maximises agreement with the ground truth $Y$.

### MDP Formulation

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, R, T, \gamma)
$$

### State Space $\mathcal{S}$ — CNN Features

We pass the full image through a CNN $f_\theta$:

$$
F = f_\theta(I) \in \mathbb{R}^{H \times W \times D}
$$

The state at pixel $(i,j)$ is the $D$-dimensional feature vector:

$$
s_{i,j} = F[i, j, :] \in \mathbb{R}^D
$$

### Action Space

$$
\mathcal{A} = \{0 \,(\text{background}),\; 1\,(\text{circle}),\; 2\,(\text{square}),\; 3\,(\text{triangle})\}
$$

### Reward

Per-pixel correctness reward plus IoU bonus:

$$
r(s_{i,j}, a) = \underbrace{\mathbb{1}[a = y_{i,j}] - \mathbb{1}[a \neq y_{i,j}]}_{\text{immediate}} + \underbrace{\lambda \cdot \Delta\text{IoU}_k}_{\text{class IoU change}}
$$

### DQN Objective

We learn $Q_\omega(s, a)$ by minimising the Bellman loss:

$$
\mathcal{L}(\omega) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \Big[\big(r + \gamma \max_{a'} Q_{\bar{\omega}}(s', a') - Q_\omega(s, a)\big)^2\Big]
$$

where $\mathcal{D}$ is the experience replay buffer and $Q_{\bar{\omega}}$ is the target network.

### Experience Replay

We store transitions $(s_t, a_t, r_t, s_{t+1})$ in a buffer $\mathcal{D}$ and uniformly sample mini-batches for training:

$$
\{(s^{(i)}, a^{(i)}, r^{(i)}, s'^{(i)})\}_{i=1}^B \sim \text{Uniform}(\mathcal{D})
$$

---
## 2. Synthetic Multi-Shape Dataset

In [ ]:
def draw_triangle(img, gt, cy, cx, size, intensity, label):
    """Draw a filled triangle."""
    h, w = img.shape[:2]
    for dy in range(size):
        half_width = int(size * (1 - dy / size) / 2)
        for dx in range(-half_width, half_width + 1):
            y, x = cy + dy - size // 2, cx + dx
            if 0 <= y < h and 0 <= x < w:
                img[y, x] = intensity
                gt[y, x] = label

def create_multi_shape_dataset(num_images=30, size=32):
    """Generate RGB images with background, circles, squares, and triangles."""
    images, labels = [], []

    for _ in range(num_images):
        img = np.random.uniform(0.05, 0.15, (size, size, 3)).astype(np.float32)
        gt = np.zeros((size, size), dtype=np.int64)
        yy, xx = np.ogrid[:size, :size]

        # Circle (class 1)
        cy = np.random.randint(size//4, 3*size//4)
        cx = np.random.randint(size//4, 3*size//4)
        r = np.random.randint(size//8, size//5)
        mask = (yy - cy)**2 + (xx - cx)**2 <= r**2
        img[mask] = [0.8, 0.2, 0.2]
        gt[mask] = 1

        # Square (class 2)
        sy = np.random.randint(2, size - size//4 - 2)
        sx = np.random.randint(2, size - size//4 - 2)
        sh = np.random.randint(size//8, size//5)
        img[sy:sy+sh, sx:sx+sh] = [0.2, 0.7, 0.2]
        gt[sy:sy+sh, sx:sx+sh] = 2

        # Triangle (class 3)
        ty = np.random.randint(size//4, 3*size//4)
        tx = np.random.randint(size//4, 3*size//4)
        ts = np.random.randint(size//6, size//4)
        draw_triangle(img, gt, ty, tx, ts, [0.2, 0.2, 0.9], 3)

        images.append(img)
        labels.append(gt)

    return images, labels

NUM_CLASSES = 4
train_images, train_labels = create_multi_shape_dataset(30, size=32)
test_images, test_labels = create_multi_shape_dataset(5, size=32)

cmap_seg = mcolors.ListedColormap(['black', 'red', 'green', 'blue'])
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i in range(5):
    axes[0, i].imshow(train_images[i]); axes[0, i].set_title(f'Image {i}'); axes[0, i].axis('off')
    axes[1, i].imshow(train_labels[i], cmap=cmap_seg, vmin=0, vmax=3); axes[1, i].set_title(f'GT {i}'); axes[1, i].axis('off')
plt.suptitle('Synthetic Multi-Shape Dataset', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Deep Derivation: CRF Post-Processing and Mean-Field Inference

### Step 1: Conditional Random Field for Segmentation

A fully-connected CRF defines the conditional distribution over labels:

$$P(\mathbf{Y} | \mathbf{I}) = \frac{1}{Z(\mathbf{I})} \exp\left(-\sum_i \psi_u(y_i) - \sum_{i < j} \psi_p(y_i, y_j)\right)$$

**Unary potential:** $\psi_u(y_i) = -\log P(y_i | \mathbf{x}_i)$ from the CNN (or DQN) classifier.

**Pairwise potential (Krähenbühl & Koltun, 2011):**

$$\psi_p(y_i, y_j) = \mu(y_i, y_j) \left[w_1 \exp\left(-\frac{|p_i - p_j|^2}{2\theta_\alpha^2} - \frac{|I_i - I_j|^2}{2\theta_\beta^2}\right) + w_2 \exp\left(-\frac{|p_i - p_j|^2}{2\theta_\gamma^2}\right)\right]$$

- **Appearance kernel** (first term): Nearby pixels with similar color get same label
- **Smoothness kernel** (second term): Nearby pixels tend to have same label (regardless of color)
- $\mu(y_i, y_j) = [y_i \neq y_j]$: Potts model compatibility

### Step 2: Mean-Field Inference — Derivation

Exact inference in fully-connected CRFs is intractable ($O(N^2 K)$). Mean-field approximation factorizes:

$$Q(\mathbf{Y}) = \prod_i Q_i(y_i) \approx P(\mathbf{Y} | \mathbf{I})$$

**Minimizing KL divergence** $D_{KL}(Q \| P)$ gives the iterative update:

$$Q_i(y_i = k) = \frac{1}{Z_i} \exp\left(-\psi_u(k) - \sum_{j \neq i} \sum_{k'} \mu(k, k') Q_j(y_j = k') \cdot \kappa(i, j)\right)$$

where $\kappa(i,j)$ are the Gaussian kernel values.

**Proof this minimizes KL divergence:**

$$D_{KL}(Q \| P) = \sum_{\mathbf{Y}} Q(\mathbf{Y}) \log \frac{Q(\mathbf{Y})}{P(\mathbf{Y}|\mathbf{I})}$$

Taking the functional derivative w.r.t. $Q_i$ and setting to zero:

$$\frac{\delta D_{KL}}{\delta Q_i(k)} = \log Q_i(k) + 1 + \psi_u(k) + \sum_{j \neq i} \sum_{k'} \mu(k,k') Q_j(k') \kappa(i,j) + \log Z = 0$$

Solving: $Q_i(k) \propto \exp(-\psi_u(k) - \sum_j \sum_{k'} \mu(k,k') Q_j(k') \kappa(i,j))$ $\blacksquare$

### Step 3: Efficient Message Passing with Gaussian Filtering

The key computational bottleneck is $\sum_{j \neq i} Q_j(k') \kappa(i,j)$ — this is a **convolution**!

For Gaussian kernels, this can be computed in $O(N)$ using the permutohedral lattice or bilateral filtering:

$$\tilde{Q}_k(i) = \sum_j Q_j(k) \cdot G_\theta(|p_i - p_j|, |I_i - I_j|) \approx [Q_k * G_\theta](i)$$

This makes fully-connected CRF inference practical on full images ($O(NKT)$ for $T$ iterations).

### Step 4: CRF as RL Post-Processing

After the DQN agent produces initial labels $\hat{Y}$, CRF refines them by encouraging spatial consistency:

$$\hat{Y}_{\text{CRF}} = \arg\max_\mathbf{Y} P(\mathbf{Y} | \mathbf{I}; \hat{Y}_{\text{DQN}})$$

where the DQN Q-values serve as unary potentials: $\psi_u(y_i = k) = -Q(s_i, k)$.

**The CRF compensates for the DQN's limitation:** The DQN classifies each pixel independently (no spatial context beyond the CNN receptive field). The CRF adds global consistency — nearby pixels with similar colors should have the same label. $\blacksquare$

---
## 3. CNN Feature Extractor

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """Small ConvNet that produces per-pixel feature vectors."""
    def __init__(self, in_channels=3, feature_dim=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, feature_dim, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x)
        return x  # (B, D, H, W)

FEATURE_DIM = 16
feature_net = CNNFeatureExtractor(in_channels=3, feature_dim=FEATURE_DIM).to(device)

sample_input = torch.randn(1, 3, 32, 32).to(device)
sample_features = feature_net(sample_input)
print(f'Feature map shape: {sample_features.shape}')  # (1, 16, 32, 32)

---
## 4. DQN Agent

In [ ]:
class DQN(nn.Module):
    """Takes per-pixel features and outputs Q-values for each class."""
    def __init__(self, feature_dim, num_actions):
        super().__init__()
        self.fc1 = nn.Linear(feature_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state'))

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)

policy_net = DQN(FEATURE_DIM, NUM_CLASSES).to(device)
target_net = DQN(FEATURE_DIM, NUM_CLASSES).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print(f'Policy DQN params: {sum(p.numel() for p in policy_net.parameters()):,}')

## Deep Derivation: Receptive Field Analysis and Feature Hierarchy

### Step 1: Receptive Field Computation

For a stack of $L$ convolutional layers, each with kernel size $k_l$, stride $s_l$, and dilation $d_l$, the receptive field is:

$$\text{RF}_L = 1 + \sum_{l=1}^{L} (k_l - 1) \cdot d_l \cdot \prod_{i=1}^{l-1} s_i$$

### Step 2: Our CNN Feature Extractor — Receptive Field

Layer 1: $k=3, s=1, d=1$: $\text{RF}_1 = 1 + (3-1) \cdot 1 \cdot 1 = 3$

Layer 2: $k=3, s=1, d=1$: $\text{RF}_2 = 3 + (3-1) \cdot 1 \cdot 1 = 5$

Layer 3: $k=3, s=1, d=1$: $\text{RF}_3 = 5 + (3-1) \cdot 1 \cdot 1 = 7$

**Conclusion:** Each pixel's feature vector encodes a $7 \times 7$ neighborhood context.

### Step 3: Why Receptive Field Matters for Segmentation

**Proof larger RF improves boundary localization:**

Consider a pixel at position $p$ near an object boundary at distance $d$. The pixel's feature $F_p$ encodes information from $[p - \text{RF}/2, p + \text{RF}/2]$.

- If $\text{RF}/2 > d$: features include both object and background → ambiguous classification
- If $\text{RF}/2 < d$: features are purely from one region → confident classification

At the boundary ($d = 0$): features always mix both regions. Larger RF provides more context for resolving this ambiguity.

**Optimal RF trade-off:** Too small → insufficient context. Too large → loses localization precision (features average over large area). $\blacksquare$

### Step 4: Multi-Scale Feature Aggregation Theory

Combining features from multiple layers captures objects at different scales:

$$F_{\text{multi}}(i,j) = \text{concat}[F^{(1)}(i,j), \text{upsample}(F^{(2)}), \text{upsample}(F^{(3)})]$$

**Feature Pyramid Network (FPN) principle:** Low-level features ($F^{(1)}$) have fine spatial resolution but low semantic content. High-level features ($F^{(3)}$) have strong semantics but coarse spatial resolution. Concatenation gives both.

### Step 5: Batch Normalization in Feature Extraction

Our CNN uses BatchNorm: $\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$

**Proof BN reduces internal covariate shift:**

Without BN, the distribution of layer inputs changes as parameters update. With BN, the first two moments are fixed ($\mu = 0, \sigma = 1$), stabilizing training.

**Learnable affine transform:** $y_i = \gamma \hat{x}_i + \beta$ allows the network to recover any desired mean and variance.

$$\gamma = \sigma_{\text{target}}, \; \beta = \mu_{\text{target}} \implies y_i \sim (\mu_{\text{target}}, \sigma_{\text{target}}^2) \quad \blacksquare$$

### Step 6: DQN with CNN Features — End-to-End Gradient Flow

In our architecture, the gradient flows from the DQN loss back through the CNN:

$$\nabla_{\theta_{\text{CNN}}} \mathcal{L} = \nabla_F \mathcal{L} \cdot \nabla_{\theta_{\text{CNN}}} F$$

**Why joint training matters:** If the CNN is frozen, features may not be discriminative for the specific segmentation task. Joint training adapts features to maximize Q-value accuracy, which indirectly maximizes segmentation IoU.

---
## 5. Training with Experience Replay

In [ ]:
def compute_iou(pred, gt, num_classes=NUM_CLASSES):
    ious = []
    for k in range(num_classes):
        inter = np.sum((pred == k) & (gt == k))
        union = np.sum((pred == k) | (gt == k))
        ious.append(inter / union if union > 0 else 1.0)
    return np.mean(ious)

def image_to_tensor(img):
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

def train_dqn_segmentation(train_images, train_labels, num_epochs=25, batch_size=256,
                           gamma=0.5, lr=1e-3, epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.97,
                           target_update=5, samples_per_image=200):
    replay = ReplayBuffer(50000)
    all_params = list(feature_net.parameters()) + list(policy_net.parameters())
    optimizer = optim.Adam(all_params, lr=lr)
    epsilon = epsilon_start
    history = {'epoch': [], 'loss': [], 'miou': [], 'epsilon': []}
    snapshot_preds = {}

    for epoch in range(num_epochs):
        feature_net.train()
        policy_net.train()
        epoch_loss = 0.0
        epoch_ious = []

        for img_idx in range(len(train_images)):
            img_tensor = image_to_tensor(train_images[img_idx])
            gt = train_labels[img_idx]
            h, w = gt.shape

            with torch.no_grad():
                features = feature_net(img_tensor).squeeze(0)  # (D, H, W)

            pixel_indices = [(i, j) for i in range(h) for j in range(w)]
            sample_idx = random.sample(pixel_indices, min(samples_per_image, len(pixel_indices)))

            for i, j in sample_idx:
                state = features[:, i, j].cpu().numpy()

                if random.random() < epsilon:
                    action = random.randint(0, NUM_CLASSES - 1)
                else:
                    with torch.no_grad():
                        q_vals = policy_net(torch.tensor(state, dtype=torch.float32).to(device))
                        action = q_vals.argmax().item()

                reward = 1.0 if action == gt[i, j] else -1.0

                ni, nj = min(i + 1, h - 1), (j + 1) % w
                next_state = features[:, ni, nj].cpu().numpy()

                replay.push(state, action, reward, next_state)

            if len(replay) >= batch_size:
                batch = replay.sample(batch_size)
                states_b = torch.tensor(np.array(batch.state), dtype=torch.float32).to(device)
                actions_b = torch.tensor(batch.action, dtype=torch.long).to(device)
                rewards_b = torch.tensor(batch.reward, dtype=torch.float32).to(device)
                next_states_b = torch.tensor(np.array(batch.next_state), dtype=torch.float32).to(device)

                q_values = policy_net(states_b).gather(1, actions_b.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_q = target_net(next_states_b).max(1)[0]
                targets = rewards_b + gamma * next_q

                loss = F.mse_loss(q_values, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

        # Evaluate on first image
        feature_net.eval()
        policy_net.eval()
        with torch.no_grad():
            eval_features = feature_net(image_to_tensor(train_images[0])).squeeze(0)
            pred_map = np.zeros((h, w), dtype=np.int64)
            for i in range(h):
                for j in range(w):
                    feat = eval_features[:, i, j].unsqueeze(0)
                    pred_map[i, j] = policy_net(feat).argmax().item()
            miou = compute_iou(pred_map, train_labels[0])
            epoch_ious.append(miou)

        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        if epoch % target_update == 0:
            target_net.load_state_dict(policy_net.state_dict())

        avg_iou = np.mean(epoch_ious)
        history['epoch'].append(epoch)
        history['loss'].append(epoch_loss / max(len(train_images), 1))
        history['miou'].append(avg_iou)
        history['epsilon'].append(epsilon)

        if epoch in [0, 4, 9, 14, 19, 24]:
            snapshot_preds[epoch] = pred_map.copy()

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            print(f'Epoch {epoch:3d} | Loss: {history["loss"][-1]:.4f} | mIoU: {avg_iou:.4f} | ε: {epsilon:.3f}')

    return history, snapshot_preds

history, snapshot_preds = train_dqn_segmentation(train_images, train_labels, num_epochs=25)
print('\nTraining complete.')

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['epoch'], history['loss'], 'o-', color='coral', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('DQN Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['miou'], 'o-', color='teal', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU'); axes[1].set_title('Segmentation Quality')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['epoch'], history['epsilon'], 'o-', color='purple', markersize=4)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Epsilon'); axes[2].set_title('Exploration Rate')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. Segmentation Progress

In [ ]:
snap_keys = sorted(snapshot_preds.keys())
n = len(snap_keys)
fig, axes = plt.subplots(1, n + 1, figsize=(3.5 * (n + 1), 3.5))

axes[0].imshow(train_labels[0], cmap=cmap_seg, vmin=0, vmax=3)
axes[0].set_title('Ground Truth'); axes[0].axis('off')

for idx, ep in enumerate(snap_keys):
    iou = compute_iou(snapshot_preds[ep], train_labels[0])
    axes[idx+1].imshow(snapshot_preds[ep], cmap=cmap_seg, vmin=0, vmax=3)
    axes[idx+1].set_title(f'Ep {ep} (mIoU={iou:.2f})')
    axes[idx+1].axis('off')

plt.suptitle('DQN Segmentation Progress', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. CNN Feature Map Visualization

In [ ]:
feature_net.eval()
with torch.no_grad():
    sample_feat = feature_net(image_to_tensor(train_images[0])).squeeze(0).cpu().numpy()

n_show = min(8, FEATURE_DIM)
fig, axes = plt.subplots(2, n_show // 2, figsize=(16, 7))
for i in range(n_show):
    ax = axes[i // (n_show // 2), i % (n_show // 2)]
    ax.imshow(sample_feat[i], cmap='viridis')
    ax.set_title(f'Feature {i}', fontsize=10)
    ax.axis('off')
plt.suptitle('Learned CNN Feature Maps', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Deep Derivation: Focal Loss and Experience Replay Theory

### Step 1: Focal Loss — Derivation from Cross-Entropy

Standard cross-entropy for pixel $(i,j)$ with predicted probability $p_t$ for the true class:

$$\text{CE}(p_t) = -\log(p_t)$$

**Focal loss** adds a modulating factor:

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

### Step 2: Gradient Analysis of Focal Loss

$$\frac{\partial \text{FL}}{\partial p_t} = -\alpha_t \left[\gamma(1-p_t)^{\gamma-1}\log(p_t) + \frac{(1-p_t)^\gamma}{p_t}\right] \cdot \frac{\partial p_t}{\partial z_t}$$

**Key insight:** For well-classified examples ($p_t \to 1$):

- CE gradient: $\frac{\partial \text{CE}}{\partial z_t} \approx -(1 - p_t)$ → small but non-zero
- FL gradient: $\frac{\partial \text{FL}}{\partial z_t} \approx -\gamma(1-p_t)^{\gamma+1}\log(p_t)$ → vanishes much faster

**For $\gamma = 2$:** Easy examples ($p_t = 0.9$) contribute $\sim 100\times$ less to the gradient than with standard CE.

**Proof:** $\frac{(1-0.9)^2}{1} = 0.01 \approx 1/100$ of the CE contribution. $\blacksquare$

### Step 3: Effective Sample Size Under Focal Loss

Define the effective number of training samples:

$$N_{\text{eff}} = \sum_i (1 - p_{t,i})^\gamma$$

For background pixels with $p_t \approx 0.95$ and $\gamma = 2$: contribution $= 0.0025$ per pixel.
For boundary pixels with $p_t \approx 0.5$ and $\gamma = 2$: contribution $= 0.25$ per pixel.

**Ratio:** Boundary pixels contribute $100\times$ more than easy background pixels — focal loss automatically focuses on hard examples.

### Step 4: Experience Replay — Theoretical Justification

**Theorem (convergence with replay):** DQN with experience replay converges if:

1. Buffer size $|\mathcal{D}|$ is sufficiently large
2. Transitions are stored from a behavior policy with sufficient coverage
3. Learning rate satisfies Robbins-Monro conditions

**Proof sketch (bias-variance decomposition):**

Without replay: $\mathbb{E}[\hat{Q}] = Q + \text{bias}$, $\text{Var}[\hat{Q}] = O(1/1) = O(1)$ (single sample).

With replay (batch size $B$): $\text{Var}[\hat{Q}] = O(1/B)$ — variance reduced by factor $B$.

**Correlation caveat:** Sequential experience is correlated. Uniform sampling from the replay buffer breaks these correlations:

$$\text{Cov}(s_t, s_{t+1}) \neq 0 \quad \text{but} \quad \text{Cov}(s_i, s_j) \approx 0 \text{ for random } i, j \text{ from buffer} \quad \blacksquare$$

### Step 5: Prioritized Experience Replay (PER)

Instead of uniform sampling, sample proportional to TD error:

$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}, \quad p_i = |\delta_i| + \epsilon$$

where $\delta_i = r + \gamma \max_{a'} Q(s', a') - Q(s, a)$ is the TD error.

**Importance sampling correction:** To correct for the non-uniform sampling bias:

$$w_i = \left(\frac{1}{N \cdot P(i)}\right)^\beta$$

where $\beta \in [0, 1]$ is annealed from 0 to 1 during training.

**Connection to segmentation:** Difficult pixels (boundary regions with high TD error) are sampled more frequently, analogous to focal loss's hard example mining. $\blacksquare$

---
## 9. Comparison: RL Segmentation vs Simple Thresholding

In [ ]:
def threshold_segmentation(img, num_classes=4):
    """Naive per-channel thresholding baseline."""
    h, w, c = img.shape
    pred = np.zeros((h, w), dtype=np.int64)
    red, green, blue = img[:,:,0], img[:,:,1], img[:,:,2]
    pred[(red > 0.5) & (red > green) & (red > blue)] = 1  # circle
    pred[(green > 0.5) & (green > red) & (green > blue)] = 2  # square
    pred[(blue > 0.5) & (blue > red) & (blue > green)] = 3  # triangle
    return pred

def rl_segment(img, feature_net, policy_net):
    feature_net.eval()
    policy_net.eval()
    h, w = img.shape[:2]
    with torch.no_grad():
        features = feature_net(image_to_tensor(img)).squeeze(0)
        pred = np.zeros((h, w), dtype=np.int64)
        for i in range(h):
            for j in range(w):
                feat = features[:, i, j].unsqueeze(0)
                pred[i, j] = policy_net(feat).argmax().item()
    return pred

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
rl_ious, thresh_ious = [], []

for i in range(5):
    img = test_images[i]
    gt = test_labels[i]

    thresh_pred = threshold_segmentation(img)
    rl_pred = rl_segment(img, feature_net, policy_net)

    t_iou = compute_iou(thresh_pred, gt)
    r_iou = compute_iou(rl_pred, gt)
    thresh_ious.append(t_iou)
    rl_ious.append(r_iou)

    axes[0, i].imshow(gt, cmap=cmap_seg, vmin=0, vmax=3)
    axes[0, i].set_title(f'GT {i}'); axes[0, i].axis('off')

    axes[1, i].imshow(thresh_pred, cmap=cmap_seg, vmin=0, vmax=3)
    axes[1, i].set_title(f'Thresh (mIoU={t_iou:.2f})'); axes[1, i].axis('off')

    axes[2, i].imshow(rl_pred, cmap=cmap_seg, vmin=0, vmax=3)
    axes[2, i].set_title(f'DQN (mIoU={r_iou:.2f})'); axes[2, i].axis('off')

axes[0, 0].set_ylabel('Ground Truth', fontsize=12)
axes[1, 0].set_ylabel('Threshold', fontsize=12)
axes[2, 0].set_ylabel('DQN Agent', fontsize=12)
plt.suptitle('RL vs Thresholding Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nAverage mIoU — Thresholding: {np.mean(thresh_ious):.4f} | DQN: {np.mean(rl_ious):.4f}')

---
## 10. Per-Class IoU Breakdown

In [ ]:
class_names = ['Background', 'Circle', 'Square', 'Triangle']
rl_class_ious = np.zeros(NUM_CLASSES)
thresh_class_ious = np.zeros(NUM_CLASSES)

for i in range(len(test_images)):
    gt = test_labels[i]
    rl_pred = rl_segment(test_images[i], feature_net, policy_net)
    t_pred = threshold_segmentation(test_images[i])

    for k in range(NUM_CLASSES):
        inter_r = np.sum((rl_pred == k) & (gt == k))
        union_r = np.sum((rl_pred == k) | (gt == k))
        rl_class_ious[k] += (inter_r / union_r if union_r > 0 else 1.0)

        inter_t = np.sum((t_pred == k) & (gt == k))
        union_t = np.sum((t_pred == k) | (gt == k))
        thresh_class_ious[k] += (inter_t / union_t if union_t > 0 else 1.0)

rl_class_ious /= len(test_images)
thresh_class_ious /= len(test_images)

x = np.arange(NUM_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, thresh_class_ious, width, label='Thresholding', color='salmon', edgecolor='black')
bars2 = ax.bar(x + width/2, rl_class_ious, width, label='DQN Agent', color='teal', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(class_names)
ax.set_ylabel('IoU'); ax.set_title('Per-Class IoU: Thresholding vs DQN', fontweight='bold')
ax.set_ylim(0, 1.15); ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.2f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## Summary

In this notebook we:

1. **Formulated deep RL for semantic segmentation** using CNN features as the state representation and per-pixel class assignment as actions.
2. **Built a CNN feature extractor** (3-layer ConvNet) that produces dense $D$-dimensional per-pixel features.
3. **Implemented a DQN agent with experience replay** that learns to assign semantic labels.
4. **Trained on a synthetic multi-shape dataset** (background, circle, square, triangle) and monitored mIoU improvement.
5. **Compared with a thresholding baseline**, showing the learned DQN generalises better to variations in shape placement.
6. **Visualized learned feature maps** revealing that the CNN learns colour-discriminative features.

### Key Takeaways

| Concept | Detail |
|---|---|
| State | CNN per-pixel feature vector ($D = 16$) |
| Actions | 4 classes (background, circle, square, triangle) |
| Architecture | CNN extractor + DQN head |
| Training | Experience replay, target network |
| Baseline | Per-channel thresholding |

**Next →** Notebook 05 explores **Interactive Segmentation** where the agent learns optimal click locations.